# Lab 0-04 Walkthrough: What Changes When a Model Becomes an Agent?

In this notebook, you will run the **same model** twice on the same mini device-activity case packet.

Run 1 is a plain model prompt.
Run 2 wraps the same model in the **Device Activity Summary Agent** workflow with:
- a limited role
- approved tools
- short memory notes
- a stop condition
- a required output format

The goal is to see that an agent is not just a model with a fancy name. In this course, an agent is a model placed inside a structured workflow.


In [ ]:
import csv
import json
from pathlib import Path
from time import perf_counter

from dotenv import dotenv_values
from openai import OpenAI

# Teaching note:
# This setup cell assumes you opened the notebook from this lab folder, then
# loads this lab's .env and the local mini case packet.
LAB_NAME = 'lab0_04_what_is_an_agent'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError(f'Could not find {LAB_NAME}/data')

config = dotenv_values(env_path)
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print("Default model from this lab's .env:", default_model)
print('Ollama endpoint:', ollama_base_url)


## Step 1: Load the Mini Case Packet

This packet is intentionally small.

It gives the model the same kind of ingredients you will use in the later labs:
- a case brief
- an artifact manifest
- a small event log

In this notebook, the local Python functions that read those files act as simple approved tools.


In [ ]:
def read_markdown(path: Path) -> str:
    return path.read_text(encoding='utf-8').strip()


def read_json_file(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


# Teaching note:
# `read_csv_rows(...)` turns the CSV event log into Python rows.
# That gives the notebook a simple local data source it can reuse like a tiny tool input.
def read_csv_rows(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))


case_brief = read_markdown(data_dir / 'case_brief.md')
artifact_manifest = read_json_file(data_dir / 'artifact_manifest.json')
device_events = read_csv_rows(data_dir / 'triage_events.csv')


def format_rows(rows: list[dict]) -> str:
    return json.dumps(rows, indent=2)


def read_case_brief() -> str:
    return case_brief


def read_manifest() -> str:
    return json.dumps(artifact_manifest, indent=2)


def read_device_events() -> str:
    return format_rows(device_events)


toolbox = {
    'read_case_brief': read_case_brief,
    'read_manifest': read_manifest,
    'read_device_events': read_device_events,
}

case_packet = '\n\n'.join([
    'CASE BRIEF\n' + read_case_brief(),
    'ARTIFACT MANIFEST\n' + read_manifest(),
    'DEVICE EVENTS\n' + read_device_events(),
])

print('Case ID:', artifact_manifest['case_id'])
print('Case question:', artifact_manifest['question'])
print('\nApproved notebook tools:')
for name in toolbox:
    print('-', name)
print('\nLoaded device events:', len(device_events))
print('\nCase brief preview:\n')
print(case_brief)


## Step 2: Ask the Same Model a Plain Question

This first run uses the full case packet, but the prompt is still open-ended.

There is no explicit role, no memory, no stop rule, and no required output schema.


In [ ]:
plain_prompt = f"""
Review the mini device-activity case packet below.

Explain:
1. What the events show.
2. What is still unknown.
3. What a human reviewer should check next.

Case packet:
{case_packet}
""".strip()


# Teaching note:
# `ask_model(...)` is the step that sends one prompt to the configured model.
# It also captures timing and raw text so students can compare different workflow styles.
def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}],
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
    }


plain_response = ask_model(default_model, plain_prompt)

print('Plain prompt preview:\n')
print(plain_prompt[:2000])
print('\nPlain-model response:')
print('Model:', plain_response['model'])
print('Time (seconds):', plain_response['seconds'])
print('-' * 80)
print(plain_response['raw_text'])


## Step 3: Wrap the Same Model in the Device Activity Summary Agent

Now you will keep the model the same but add agent components around it.

This walkthrough agent has:
- a limited role
- a concrete goal
- approved tools
- short memory notes
- a stop condition
- a required JSON output


In [ ]:
agent_spec = {
    'agent_name': 'Device Activity Summary Agent',
    'role': 'Produce a cautious first-pass summary of a small device-activity packet.',
    'goal': 'List clear observations, note what is still unknown, recommend one next review step, and stop.',
    'approved_tools': list(toolbox.keys()),
    'memory': [
        'This packet is small and may be incomplete.',
        'A created file, message text, network event, and deletion do not prove who saw the file.',
        'A human reviewer should confirm any stronger interpretation.',
    ],
    'required_output_keys': [
        'observations',
        'unknowns',
        'next_step',
        'needs_human_review',
        'why_this_is_agent_behavior',
    ],
    'stop_condition': 'Stop after returning the required JSON object.',
}

print('Agent spec:\n')
print(json.dumps(agent_spec, indent=2))


# Teaching note:
# `build_agent_prompt(...)` turns the agent specification fields into one bounded prompt.
# This is where role, memory, rules, and stop condition become agent workflow structure.
def build_agent_prompt(agent: dict, packet: str) -> str:
    tool_lines = '\n'.join(f'- {name}' for name in agent['approved_tools'])
    memory_lines = '\n'.join(f'- {note}' for note in agent['memory'])
    required_keys = ', '.join(agent['required_output_keys'])
    return f"""
You are {agent['agent_name']}.

Role:
{agent['role']}

Goal:
{agent['goal']}

Approved tools:
{tool_lines}

Memory:
{memory_lines}

Rules:
- Use only the information from the provided case packet.
- Do not claim that the screenshot contents or delivery are confirmed unless the packet directly confirms them.
- Keep observations separate from unknowns.
- Recommend exactly one next review step.
- Return valid JSON only.
- Use exactly these keys: {required_keys}.

Stop condition:
{agent['stop_condition']}

Case packet:
{packet}
""".strip()


# Teaching note:
# Models sometimes wrap JSON in code fences or add extra text around it.
# `clean_json_text(...)` trims that noise so the notebook can parse the response more reliably.
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned


agent_prompt = build_agent_prompt(agent_spec, case_packet)
agent_response = ask_model(default_model, agent_prompt)

print('\nAgent prompt preview:\n')
print(agent_prompt[:2200])
print('\nAgent response:')
print('Model:', agent_response['model'])
print('Time (seconds):', agent_response['seconds'])
print('-' * 80)
print(agent_response['raw_text'])

agent_json = None
try:
    agent_json = json.loads(clean_json_text(agent_response['raw_text']))
except Exception as exc:
    print('\nAgent JSON parse error:', exc)

if agent_json is not None:
    print('\nParsed agent JSON:')
    print(json.dumps(agent_json, indent=2))


## Step 4: Compare What Changed

The model itself did not change.

What changed was the workflow around the model.
That workflow gave the model:
- a narrower job
- limited inputs
- explicit constraints
- a memory summary
- a stop point
- a structured output requirement


In [ ]:
comparison = {
    'same_model_used_for_both_runs': default_model,
    'plain_prompt_has_explicit_role': False,
    'agent_prompt_has_explicit_role': True,
    'agent_prompt_has_approved_tools': True,
    'agent_prompt_has_memory': True,
    'agent_prompt_has_stop_condition': True,
    'agent_prompt_has_required_output_schema': True,
}

print(json.dumps(comparison, indent=2))

print('\nQuick comparison notes:')
print('- Plain-model response is open-ended free text.')
print('- Agent response is bounded by role, tools, memory, and a stop rule.')
print('- The same model can behave differently when the workflow around it changes.')

if agent_json is not None:
    print('\nAgent output keys:', sorted(agent_json.keys()))


## Reflection Questions

Replace this text with short answers to the questions below.

1. What is the biggest difference you noticed between the plain-model response and the agent response?
2. Which agent component seemed most important: role, tools, memory, stop condition, or output schema?
3. Did the agent stay more bounded than the plain prompt? Give one example.
4. Why does this notebook support the idea that an agent is a workflow, not just a model name?

## Submission

Save the notebook with:
- the loaded case packet output
- the plain-model response
- the agent specification
- the agent response
- your short reflection answers
